# Week 1. Word Embeddings — семинар NLP (ВШЭ)

**Программа:** магистратура / курс NLP (Высшая школа экономики)  
**Тема:** токенизация, Word2Vec / FastText / GloVe, визуализация эмбеддингов (PCA, t-SNE)

Ноутбук оформлен после решения семинара: добавлены мои комментарии, мини-тесты на OOV и заметки по методологии (скейлинг до PCA, различие архитектур, чтение графиков).





## 0. Окружение

Локально: `pip install --upgrade nltk gensim bokeh scikit-learn numpy`.


In [ ]:
# %pip install --upgrade nltk gensim bokeh scikit-learn numpy
print("deps: nltk, gensim, bokeh, sklearn, numpy")


## 1. Данные Quora

Скачиваем корпус вопросов (если файла ещё нет).


In [ ]:
from pathlib import Path
import urllib.request

data_path = Path("./quora.txt")
if not data_path.exists():
    url = "https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1"
    print("downloading quora.txt ...")
    urllib.request.urlretrieve(url, data_path)
else:
    print("quora.txt already exists")

with open(data_path, encoding="utf-8") as file:
    data = list(file)

print(f"lines: {len(data)}")
data[50]


## 2. Токенизация

Сырой текст с пунктуацией — `str.split` не хватит. Используем `nltk.WordPunctTokenizer`.


In [ ]:
from nltk.tokenize import WordPunctTokenizer

tokenizer = WordPunctTokenizer()
print(tokenizer.tokenize(data[50]))


In [ ]:
# TASK: lowercase + tokenize
# data_tok — list[list[str]] для каждой строки

data_tok = [tokenizer.tokenize(row.lower()) for row in data]

assert isinstance(data_tok, list) and isinstance(data_tok[0], list)
assert all(isinstance(t, str) for t in data_tok[0])
assert data_tok[0][0].islower() or not data_tok[0][0].isalpha()
print([" ".join(row) for row in data_tok[:2]])
print(f"tokenized lines: {len(data_tok)}")


## 3. Свои эмбеддинги: FastText (gensim)

**Word2Vec / GloVe** учат вектор на целое слово из словаря.  
**FastText** дополнительно использует **символьные n-граммы** → может собрать вектор для слова, которого не было в обучении (OOV), если похожие куски символов встречались.

Ниже учу маленький FastText на Quora, потом сравню с Word2Vec по OOV.


In [ ]:
from gensim.models import FastText, Word2Vec

# FastText на нашем корпусе
ft_model = FastText(
    data_tok,
    vector_size=32,
    min_count=5,
    window=5,
    workers=4,
    seed=42,
).wv

print("vector(anything)[:5] =", ft_model.get_vector("anything")[:5])
print("'anything' in vocab:", "anything" in ft_model.key_to_index)
ft_model.most_similar("python")


### Мой комментарий: Word2Vec vs FastText на неизвестных словах

Я отдельно проверил поведение на **OOV** (out-of-vocabulary).

- **Word2Vec** (и классический GloVe-словарь): слово либо есть в `key_to_index`, либо нет. На неизвестном токене `get_vector` / обращение к модели даёт **ошибку** (`KeyError`). Для русского «кривого» слова, которого не было в корпусе и которое ни с чем не «стакается» по полному совпадению — вектора просто нет.
- **FastText**: даже если полного слова не было, модель может вернуть вектор через **сумму/усреднение char n-gram**. На практике для неизвестного русского слова часто **не падает**, а отдаёт какой-то вектор. Он не всегда «смысловой», но пайплайн не ломается — удобно для морфологически богатого русского и опечаток.

Ниже — явные тесты.


In [ ]:
# --- тесты OOV: Word2Vec vs FastText ---

w2v_model = Word2Vec(
    data_tok,
    vector_size=32,
    min_count=5,
    window=5,
    workers=4,
    seed=42,
).wv

# искусственно «редкое» / неизвестное: с низкой вероятностью попало в vocab min_count>=5
oov_candidates = [
    "абракадабровщина123",
    "нейросеткоподобный",
    "ъьыэюяzzz",
    "quorawordthatshouldnotexistxyz",
]
oov_word = next((w for w in oov_candidates if w not in w2v_model.key_to_index), oov_candidates[0])
print("test OOV token:", oov_word)
print("in Word2Vec vocab:", oov_word in w2v_model.key_to_index)
print("in FastText vocab:", oov_word in ft_model.key_to_index)

# Word2Vec → ожидаем ошибку
w2v_error = None
try:
    _ = w2v_model.get_vector(oov_word)
except KeyError as e:
    w2v_error = e
    print("Word2Vec OOV → KeyError (ожидаемо):", type(e).__name__, e)

assert w2v_error is not None, "Word2Vec должен падать на неизвестном слове"

# FastText → вектор без падения
ft_vec = ft_model.get_vector(oov_word)
assert ft_vec.shape == (32,)
print("FastText OOV → vector ok, l2 =", float((ft_vec ** 2).sum() ** 0.5))
print("most_similar to OOV (могут быть шумные соседи):")
print(ft_model.most_similar(oov_word)[:5])


In [ ]:
# немного поиграть с похожестью на обученном FastText
for q in ["love", "python", "music", "что"]:
    if q in ft_model.key_to_index or True:
        try:
            print(q, "→", ft_model.most_similar(q)[:3])
        except Exception as e:
            print(q, "failed:", e)


## 4. Pretrained: GloVe Twitter 100d

Готовая модель из `gensim.downloader` — быстрее и качественнее, чем наш крошечный FastText на Quora.


In [ ]:
import gensim.downloader as api

# скачается один раз в gensim-data
model = api.load("glove-twitter-100")
print(type(model), "dim =", model.vector_size)
print(model.most_similar("wine")[:5])
print("analogy coder - brain + money →", model.most_similar(positive=["coder", "money"], negative=["brain"])[:5])


## 5. Визуализация: PCA и t-SNE

Эмбеддинги ~100D. Людям удобнее 2D → снижение размерности.


In [ ]:
import numpy as np

words = model.index_to_key[:1000]
print("sample words:", words[::100])

word_vectors = np.stack([model.get_vector(word) for word in words], axis=0)
print("word_vectors.shape =", word_vectors.shape)

assert isinstance(word_vectors, np.ndarray)
assert word_vectors.shape == (len(words), 100)
assert np.isfinite(word_vectors).all()


### PCA

PCA ищет оси максимальной дисперсии (линейная проекция).

#### Мой комментарий про StandardScaler

**Важно:** признаки перед PCA лучше **стандартизировать до** `fit` PCA (`StandardScaler` / центрирование), а не «потом как попало».

Почему:
- PCA чувствителен к масштабу осей. Если одна координата эмбеддинга «гуляет» сильнее других, она съест первые компоненты.
- Сначала `StandardScaler` на исходных векторах → затем `PCA(n_components=2)`.
- Отдельно для **картинки** иногда ещё нормализуют уже 2D-проекцию (чтобы точки были zero-mean / unit-variance на плоскости) — это про удобство визуализации и ассерты семинара, но **методологически главный скейлинг — до PCA**.

В первой версии решения я как раз сначала сделал PCA, а scaler после — для обучения зафиксировал правильный порядок ниже.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1) standardization BEFORE PCA (правильный порядок)
scaler_raw = StandardScaler()
word_vectors_std = scaler_raw.fit_transform(word_vectors)

pca = PCA(n_components=2, random_state=42)
word_vectors_pca = pca.fit_transform(word_vectors_std)
print("explained variance ratio:", pca.explained_variance_ratio_)
print("PCA out sample:", word_vectors_pca[:1])

# 2) optional: scale 2D plane for viz / seminar-style asserts
scaler_2d = StandardScaler()
word_vectors_pca_scaled = scaler_2d.fit_transform(word_vectors_pca)

assert word_vectors_pca_scaled.shape == (len(word_vectors), 2)
assert max(abs(word_vectors_pca_scaled.mean(0))) < 1e-5
assert max(abs(1.0 - word_vectors_pca_scaled.std(0))) < 1e-2
print("PCA+scale asserts OK")


In [ ]:
import bokeh.models as bm
import bokeh.plotting as pl
from bokeh.io import output_notebook

output_notebook()

def draw_vectors(x, y, radius=10, alpha=0.25, color="blue",
                 width=600, height=400, show=True, **kwargs):
    """interactive scatter with hover"""
    if isinstance(color, str):
        color = [color] * len(x)
    data_source = bm.ColumnDataSource({"x": x, "y": y, "color": color, **kwargs})
    fig = pl.figure(active_scroll="wheel_zoom", width=width, height=height)
    fig.scatter("x", "y", size=radius, color="color", alpha=alpha, source=data_source)
    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show:
        pl.show(fig)
    return fig


In [ ]:
# рисуем отскейленную 2D-проекцию
draw_vectors(
    word_vectors_pca_scaled[:, 0],
    word_vectors_pca_scaled[:, 1],
    token=words,
)
# hover: ищи кластеры (дни недели, имена, эмоции, tech, ...)


### t-SNE

t-SNE — нелинейный метод: старается сохранить **локальных соседей**. Глобальная геометрия может «плыть» (см. [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/)).

#### Мой комментарий: PCA vs t-SNE на графиках

- **PCA:** быстрее, детерминированнее, видны грубые глобальные направления. Кластеры могут быть размытыми, но структура «крупных осей» читается. Первые компоненты объясняют долю дисперсии — это плюс для отчёта.
- **t-SNE:** локальные кучки слов (синонимы, темы) часто **ярче**. Но расстояние между далёкими кластерами интерпретировать опасно; от `perplexity` / seed картинка меняется.
- Практика: PCA — быстрый sanity-check; t-SNE — «посмотреть соседей». Для эмбеддингов иногда ещё UMAP, но в семинаре — PCA/TSNE.

На hover в t-SNE обычно лучше видны плотные тематические группки, на PCA — более вытянутое облако.


In [ ]:
from sklearn.manifold import TSNE

# t-SNE на уже стандартизованных векторах (тот же принцип: масштаб до редукции)
tsne = TSNE(n_components=2, random_state=42, init="pca", learning_rate="auto")
word_vectors_tsne = tsne.fit_transform(word_vectors_std)

scaler_tsne = StandardScaler()
word_vectors_tsne_scaled = scaler_tsne.fit_transform(word_vectors_tsne)

draw_vectors(
    word_vectors_tsne_scaled[:, 0],
    word_vectors_tsne_scaled[:, 1],
    token=words,
)


## 6. Что ещё посмотреть

- Другие модели в зоопарке: `gensim.downloader.info()`
- [FastText (Facebook)](https://github.com/facebookresearch/fastText)
- Свои корпуса / доменные эмбеддинги под задачу


In [ ]:
import pprint
info = api.info()
print("num corpora:", len(info.get("corpora", {})))
print("num models:", len(info.get("models", {})))
# короткий список моделей
pprint.pp(sorted(info["models"].keys())[:25])


## Итог (заметки для себя / для GitHub)

1. Токенизация — первый обязательный шаг на сыром тексте.  
2. **Word2Vec** падает на OOV; **FastText** часто вывозит неизвестные/кривые слова через n-граммы.  
3. Pretrained GloVe удобен для быстрых экспериментов и аналогий.  
4. **StandardScaler → PCA** (скейл до редукции); 2D-скейл — опционально для визуализации.  
5. **PCA** = глобальная линейная структура; **t-SNE** = локальные кластеры, осторожнее с интерпретацией расстояний.

Семинар week 1 прорешан, с моими заметками по OOV / PCA / t-SNE.
